In [ ]:
%run ./utils_common

In [ ]:
logger = setup_logger("TotalSqlWarehouseSpendsReporter")

In [ ]:
dbutils.widgets.text("catalog", "", "CATALOG")
dbutils.widgets.text("schema", "", "SCHEMA")
dbutils.widgets.text("overlap_days", "3", "Overlap days (min 2)")

In [ ]:
# =======================================================
# Total SQL Warehouse Spends Client
# =======================================================
# Sibling of TotalPipelineSpendsClient, and the simplest of the rollups.
# Denormalizes warehouse metadata onto the dbspend360_sql_warehouse_dbu_cost
# staging table and writes dbspend360_total_sql_warehouse_spends.
# Differences vs the pipeline rollup:
#   * NO cloud-cost join. All three warehouse types (Classic, Pro, Serverless)
#     run on Databricks-MANAGED compute: there are no customer-account VMs, no
#     resource tags to join on, and the cloud cost explorer has zero matching
#     rows (plan Q4). DBU IS the complete cost, so total_cost = databricks_cost
#     with no component to add and no cloud_cost column to carry. There is
#     consequently no cloud_cost_table param, no error_log_table, and no
#     reconciliation assertion - the pipeline rollup's whole §3.2/§3.3 cloud
#     attribution + invariant machinery has no analogue here.
#   * NO re-aggregation. Staging is already at (warehouse_id, usage_date), the
#     exact target grain, so this rollup is a projection + LEFT JOIN. The
#     pipeline rollup must collapse cluster_id away; there is no cluster_id
#     here (plan Q2: every SQL warehouse cluster_id is NULL).
#   * NO billing_origin_product in the grain. Every row is SQL by construction
#     (that is the staging filter), so there is no per-workload $ split to keep
#     exact and no workload_type / compute_mode / cost_basis derivation.
#   * metadata source: system.compute.warehouses (not system.lakeflow.pipelines),
#     SCD-collapsed on warehouse_id ALONE - warehouse_id is account-unique,
#     unlike pipeline_id which needs (workspace_id, pipeline_id).
#   * warehouse_type precedence: the system table value WINS over the
#     SKU-derived staging value (COALESCE(system, staging)). The staging value
#     is a billing-side inference; the system table is the warehouse's declared
#     configuration. Staging remains the fallback so a warehouse with no
#     snapshot row still gets a type instead of NULL.
#   * metadata_missing is computed BEFORE the COALESCE fallback on
#     warehouse_name, so it reflects the underlying snapshot state rather than
#     the post-fallback state. Three states the UI can render: active
#     (metadata_missing false, warehouse_deleted_at NULL) / deleted-but-visible
#     (false, timestamp set) / metadata-not-available (true).
#   * MERGE key: (warehouse_id, usage_date) - both NON-nullable, plain '='.
class TotalSqlWarehouseSpendsClient:

    TABLE_NAME = "dbspend360_total_sql_warehouse_spends"

    def __init__(
        self,
        audit_table: str,
        databricks_cost_table: str,
        target_table: str,
        overlap_days: int,
        logger=None,
    ):
        self.audit_table = audit_table
        self.databricks_cost_table = databricks_cost_table
        self.target_table = target_table
        self.overlap_days = overlap_days
        self.logger = logger or logging.getLogger("TotalSqlWarehouseSpendsClient")

    def _load_warehouse_snapshot(self):
        # SCD-collapse system.compute.warehouses to one row per warehouse_id
        # carrying the most-recent snapshot. QUALIFY ROW_NUMBER() OVER
        # (... ORDER BY change_time DESC) = 1 is holistically safe on tied
        # change_time (one winner per partition). PARTITION BY warehouse_id
        # alone is correct because warehouse_id is account-unique (plan §grain);
        # the pipeline equivalent needs (workspace_id, pipeline_id).
        # delete_time is non-null iff the warehouse was deleted; carry it
        # through as warehouse_deleted_at for the "Deleted YYYY-MM-DD" badge.
        # The join key is aliased to w_* (and warehouse_type to w_warehouse_type,
        # which also collides with the staging column) so no post-join column
        # shares a name. On serverless Spark Connect a shared-name column that
        # participates in an equi-join key becomes unresolvable when referenced
        # after the join, so every post-join column stays uniquely named and
        # only bare names are referenced downstream.
        # Source column names are the SYSTEM TABLE's, verified against
        # DESCRIBE TABLE system.compute.warehouses - they differ from the SQL
        # Warehouses REST API field names (the API calls these name /
        # cluster_size / creator_id / auto_stop_mins / num_clusters /
        # max_num_clusters). Only auto_stop_minutes needs renaming to match the
        # target DDL's auto_stop_mins; min_clusters / max_clusters /
        # auto_stop_minutes are already INT, matching the target types.
        return spark.sql("""
            SELECT warehouse_id AS w_warehouse_id,
                   warehouse_name,
                   warehouse_type AS w_warehouse_type,
                   warehouse_size,
                   created_by AS creator_id,
                   auto_stop_minutes AS auto_stop_mins,
                   min_clusters,
                   max_clusters,
                   delete_time AS warehouse_deleted_at
            FROM system.compute.warehouses
            QUALIFY ROW_NUMBER() OVER (
                PARTITION BY warehouse_id
                ORDER BY change_time DESC) = 1
        """)

    def build_total_sql_warehouse_spends(self):
        start_dt = end_dt = datetime.now(timezone.utc).date()
        try:
            start_dt, end_dt = get_date_window(self.audit_table, self.TABLE_NAME, self.overlap_days)

            valid, msg = validate_date_window(start_dt, end_dt)
            if not valid:
                raise DataQualityError(msg)

            self.logger.info(
                f"Building dbspend360_total_sql_warehouse_spends for {start_dt} → {end_dt}"
            )

            staging_df = (
                spark.table(self.databricks_cost_table)
                    .filter(
                        (F.col("usage_date") >= F.lit(start_dt)) &
                        (F.col("usage_date") <= F.lit(end_dt))
                    )
            )

            if staging_df.limit(1).count() == 0:
                self.logger.info(
                    "No SQL warehouse DBU rows in this date window; nothing to roll up."
                )
                log_audit_run(
                    self.audit_table, self.TABLE_NAME, start_dt, end_dt,
                    "SUCCESS", 0, "No DBU data in window",
                )
                return

            # 1) Project staging. NO groupBy: staging is already at the target
            #    grain (warehouse_id, usage_date). The SKU-derived type is
            #    renamed stg_warehouse_type so it never collides with the
            #    system table's warehouse_type after the join below.
            day_df = staging_df.select(
                F.col("warehouse_id"),
                F.col("usage_date"),
                F.col("databricks_cost"),
                F.col("currency"),
                F.col("sku_name"),
                F.col("warehouse_type").alias("stg_warehouse_type"),
                F.col("workspace_id"),
                F.col("workspace_covered"),
            )

            # 2) SCD-collapse system.compute.warehouses and LEFT-join metadata.
            #    LEFT so warehouse-days with no snapshot row still land in the
            #    rollup; metadata_missing is the signal (set BEFORE the COALESCE
            #    fallback paints a synthetic warehouse_name).
            warehouses_df = self._load_warehouse_snapshot()

            joined = (
                day_df
                .join(
                    warehouses_df,
                    on=(F.col("warehouse_id") == F.col("w_warehouse_id")),
                    how="left",
                )
                .withColumn("metadata_missing", F.col("warehouse_name").isNull())
            )

            select_cols = [
                F.col("warehouse_id"),
                F.col("usage_date"),
                F.coalesce(
                    F.col("warehouse_name"),
                    F.concat(F.lit("Warehouse "), F.col("warehouse_id")),
                ).alias("warehouse_name"),
                # System table value wins; the SKU-derived staging value is the
                # fallback so a warehouse with no snapshot row still has a type.
                # The system table is also the only source that can report
                # REAL_TIME - the staging SKU rules only ever emit SERVERLESS /
                # PRO / CLASSIC - so the UI's type badge map must cover it.
                F.coalesce(
                    F.col("w_warehouse_type"),
                    F.col("stg_warehouse_type"),
                ).alias("warehouse_type"),
                F.col("warehouse_size"),
                F.col("creator_id"),
                F.col("auto_stop_mins"),
                F.col("min_clusters"),
                F.col("max_clusters"),
                F.col("metadata_missing"),
                F.col("warehouse_deleted_at"),
                F.col("databricks_cost"),
                F.col("currency"),
                F.col("sku_name"),
                F.col("workspace_id"),
                F.col("workspace_covered"),
            ]

            final_df = joined.select(*select_cols)

            final_df = (
                final_df
                # DBU is the complete cost for managed compute, so total_cost is
                # a literal copy of databricks_cost - no COALESCE over a cloud
                # component like the pipeline rollup needs (plan Q4).
                .withColumn("total_cost", F.col("databricks_cost"))
                .withColumn("created_at", F.current_timestamp())
                .withColumn("updated_at", F.current_timestamp())
            )
            final_df = safe_cache(final_df)

            row_count = final_df.count()

            validate_source_schema(
                final_df,
                {"warehouse_id": "string", "usage_date": "date",
                 "warehouse_name": "string", "warehouse_type": "string",
                 "metadata_missing": "boolean", "databricks_cost": "double",
                 "total_cost": "double"},
                self.target_table, self.logger,
            )
            validate_no_negative_costs(
                final_df, ["databricks_cost", "total_cost"], self.target_table, self.logger,
            )
            validate_currency_consistency(final_df, "currency", self.target_table, self.logger)

            ensure_boolean_columns(self.target_table, ["workspace_covered"], logger=self.logger)
            target = DeltaTable.forName(spark, self.target_table)
            (target.alias("t")
                .merge(
                    final_df.alias("s"),
                    # Both key columns are NON-nullable, so plain '=' is correct.
                    "t.warehouse_id = s.warehouse_id "
                    "AND t.usage_date = s.usage_date",
                )
                .whenMatchedUpdate(set={
                    "warehouse_name": "s.warehouse_name",
                    "warehouse_type": "s.warehouse_type",
                    "warehouse_size": "s.warehouse_size",
                    "creator_id": "s.creator_id",
                    "auto_stop_mins": "s.auto_stop_mins",
                    "min_clusters": "s.min_clusters",
                    "max_clusters": "s.max_clusters",
                    "metadata_missing": "s.metadata_missing",
                    "warehouse_deleted_at": "s.warehouse_deleted_at",
                    "databricks_cost": "s.databricks_cost",
                    "total_cost": "s.total_cost",
                    "currency": "s.currency",
                    "sku_name": "s.sku_name",
                    "workspace_id": "s.workspace_id",
                    "workspace_covered": "s.workspace_covered",
                    "updated_at": "current_timestamp()",
                })
                .whenNotMatchedInsert(values={
                    "warehouse_id": "s.warehouse_id",
                    "usage_date": "s.usage_date",
                    "warehouse_name": "s.warehouse_name",
                    "warehouse_type": "s.warehouse_type",
                    "warehouse_size": "s.warehouse_size",
                    "creator_id": "s.creator_id",
                    "auto_stop_mins": "s.auto_stop_mins",
                    "min_clusters": "s.min_clusters",
                    "max_clusters": "s.max_clusters",
                    "metadata_missing": "s.metadata_missing",
                    "warehouse_deleted_at": "s.warehouse_deleted_at",
                    "databricks_cost": "s.databricks_cost",
                    "total_cost": "s.total_cost",
                    "currency": "s.currency",
                    "sku_name": "s.sku_name",
                    "workspace_id": "s.workspace_id",
                    "workspace_covered": "s.workspace_covered",
                    "created_at": "current_timestamp()",
                    "updated_at": "current_timestamp()",
                })
                .execute()
            )

            safe_unpersist(final_df)
            get_merge_metrics(self.target_table, self.logger)

            validate_post_merge(
                self.target_table, "usage_date",
                start_dt, end_dt, row_count, self.logger,
            )

            log_audit_run(
                self.audit_table, self.TABLE_NAME, start_dt, end_dt,
                "SUCCESS", row_count, "",
            )
            self.logger.info(
                f"Merged {row_count} rows into {self.target_table} "
                f"for {start_dt} → {end_dt}."
            )

        except Exception as e:
            msg = str(e)[:1000]
            self.logger.error(f"Run failed: {msg}")
            try:
                log_audit_run(
                    self.audit_table, self.TABLE_NAME, start_dt, end_dt, "FAILED", 0, msg,
                )
            except Exception:
                self.logger.error("Failed to write FAILED audit entry")
            raise

In [ ]:
# =======================================================
# APP
# =======================================================
class TotalSqlWarehouseSpendsApp:

    def __init__(self):
        catalog = dbutils.widgets.get("catalog")
        schema = dbutils.widgets.get("schema")
        ov_days = get_overlap_days(dbutils.widgets.get("overlap_days"), logger=logger)

        self.client = TotalSqlWarehouseSpendsClient(
            audit_table=build_table_fqn(catalog, schema, "dbspend360_audit_log"),
            databricks_cost_table=build_table_fqn(catalog, schema, "dbspend360_sql_warehouse_dbu_cost"),
            target_table=build_table_fqn(catalog, schema, "dbspend360_total_sql_warehouse_spends"),
            overlap_days=ov_days,
            logger=logger,
        )

    def run(self):
        self.client.build_total_sql_warehouse_spends()

In [ ]:
# =======================================================
# Execute
# =======================================================
app = TotalSqlWarehouseSpendsApp()
app.run()